# Griffin (pretrained single-sft, fine-tuned) vs SAP-RPT-1.5 on the same 180 test transactions
Inputs
- `data_creditcard_with_class_more_testcases.csv` (the file used for the SAP run)
- `sap_data_prediction_more_tests.csv` (SAP-1.5 output from your earlier notebook)
- one or more Griffin run folders produced by `griffin_fraud_run.py` (each has `predictions_test.csv`)

Griffin predictions are matched to the SAP ones by `row_id` (= row number in the original CSV), so both are scored on exactly the same rows.

In [ ]:
import json, os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, average_precision_score)

DATA = "data_creditcard_with_class_more_testcases.csv"
SAP  = "sap_data_prediction_more_tests.csv"

# name -> run folder. Add/remove entries freely (e.g. the zero-shot run and the fine-tuned run).
RUNS = {
    "Griffin zero-shot (no fine-tuning)": "runs/zeroshot",
    "Griffin fine-tuned":                 "runs/finetuned",
}

orig = pd.read_csv(DATA)
test_mask = orig["Class"].astype(str).str.strip() == "[PREDICT]"
truth = orig.loc[test_mask, "ActualClass"].astype(int)           # index = row_id
sap_pred = pd.read_csv(SAP).loc[test_mask, "Class"].astype(int)
print("test rows:", len(truth), "| fraud:", int(truth.sum()))

In [ ]:
def metrics(y, p):
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
                recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0),
                TN=tn, FP=fp, FN=fn, TP=tp)

preds = {"SAP-RPT-1.5": sap_pred.values}
extra = {}
for name, folder in RUNS.items():
    f = os.path.join(folder, "predictions_test.csv")
    if not os.path.exists(f):
        print("skip (not found):", f); continue
    g = pd.read_csv(f).set_index("row_id").loc[truth.index]
    assert (g["actual"].values == truth.values).all(), "row alignment problem"
    preds[f"{name} [argmax]"]   = g["pred_argmax"].values
    preds[f"{name} [val-thr]"]  = g["pred_valthr"].values
    extra[name] = dict(roc_auc=roc_auc_score(truth, g["prob_fraud"]), pr_auc=average_precision_score(truth, g["prob_fraud"]))

table = pd.DataFrame({k: metrics(truth.values, v) for k, v in preds.items()}).T
table.round(4)

`[argmax]` = predicted class is simply the more likely one (what SAP's UI gives you).
`[val-thr]` = fraud probability threshold picked on the validation rows (best F1) - a legitimate extra, because it never looks at the test labels.
Report `[argmax]` as the headline for a like-for-like comparison with SAP.

In [ ]:
pd.DataFrame(extra).T.round(4) if extra else print("no Griffin runs found")

In [ ]:
n = len(preds)
fig, axes = plt.subplots(1, n, figsize=(3.6 * n, 3.6))
for ax, (name, p) in zip(np.atleast_1d(axes), preds.items()):
    ConfusionMatrixDisplay(confusion_matrix(truth.values, p, labels=[0, 1]), display_labels=["Normal", "Fraud"]).plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Training curve of the fine-tuned run (validation AUROC per epoch)
for name, folder in RUNS.items():
    f = os.path.join(folder, "run_summary.json")
    if os.path.exists(f):
        h = pd.DataFrame(json.load(open(f))["history"])
        if len(h) > 2:
            plt.plot(h["epoch"], h["val_auroc"], label=name)
plt.xlabel("epoch (-1 = pretrained)"); plt.ylabel("validation AUROC"); plt.legend(); plt.show()

In [ ]:
# Which frauds does each model miss?
rows = pd.DataFrame({"actual": truth.values}, index=truth.index)
for k, v in preds.items():
    rows[k] = v
rows[rows["actual"] == 1].sort_index()

## Caveats to state in the write-up
- The test set is 1 fraud per 5 normal rows (30/180); real data is ~0.17% fraud, so precision is optimistic for every model.
- 30 fraud cases: one case is 3.3 points of recall. Do not over-read small differences; run 3+ seeds of the Griffin fine-tune (`--seed`).
- SAP-1.5 is in-context (no training). Griffin here is **fine-tuned** on ~1,570 of the 1,850 context rows (the rest is its validation set). Different protocols, say so.
- Griffin is built for multi-table databases; this is a single flat table with anonymised columns (V1-V28), so its main strengths (graph structure, column-name semantics) are largely unused.